The script below adds metadata to the pages containing transcriptions of the Council of Policy and the Orphan Chamber, to make them better discoverable by search indexes.

In [ ]:
import os
import re
from pathlib import Path

# --- CONFIGURATION ---
SOURCE_DIRS = [
    "/path/to/Council-of-Policy",
    "/path/to/Orphan-Chamber/MOOC8"
]

OUTPUT_BASE = Path("/path/to/output-folder")

DATE_RANGES = {
    "Council-of-Policy": "1651-1795",
    "Orphan-Chamber": "1690-1840"
}


def extract_document_id_from_filename(filename):
    match = re.search(r'(C\d{3,4}|MOOC\d+)', filename)
    return match.group(1) if match else None


def detect_collection_type(path):
    if "Council-of-Policy" in path:
        return "Council-of-Policy"
    if "Orphan-Chamber" in path:
        return "Orphan-Chamber"
    return None


def read_markdown_body(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    # Strip existing front matter if present
    if content.startswith("---"):
        parts = content.split("---", 2)
        if len(parts) == 3:
            return parts[2].lstrip()
    return content


def generate_front_matter(doc_id, collection):
    if "Council" in collection:
        title = f"Council of Policy Minutes – {doc_id}"
    else:
        title = f"Orphan Chamber Record – {doc_id}"

    description = f"Transcription of {title} from the Cape of Good Hope."
    date = DATE_RANGES.get(collection, "")
    location = "Cape of Good Hope"
    language = "nl"

    front_matter = f"""---
title: "{title}"
description: "{description}"
document_id: "{doc_id}"
date: {date}
location: "{location}"
language: "{language}"

schema:
  "@context": "https://schema.org"
  "@type": "CreativeWork"
  about: "VOC transcription"
  spatialCoverage: "{location}"
  inLanguage: "{language}"
  dateCreated: "{date}"
---
"""
    return front_matter


def write_with_front_matter(dest_path, front_matter, body):
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    with open(dest_path, "w", encoding="utf-8") as f:
        f.write(front_matter)
        f.write("\n")
        f.write(body)


def process_file(src_file):
    relative_path = Path(src_file).relative_to(*src_file.parts[:7])  # trims fixed segments
    dest_file = OUTPUT_BASE / relative_path

    doc_id = extract_document_id_from_filename(src_file.name)
    if not doc_id:
        print(f"⚠ Skipping (no doc ID): {src_file}")
        return

    collection = detect_collection_type(str(src_file))
    front_matter = generate_front_matter(doc_id, collection)
    body = read_markdown_body(src_file)
    write_with_front_matter(dest_file, front_matter, body)
    print(f"✔ Processed: {relative_path}")


def main():
    for folder in SOURCE_DIRS:
        for root, dirs, files in os.walk(folder):
            for file in files:
                if file.endswith(".md"):
                    process_file(Path(root) / file)


if __name__ == "__main__":
    main()
